# Objective
Assess whether the company's rapid client growth (2021–2025) is profitable and sustainable, by analysing revenue/cost trends, client segments, and satisfaction drivers to recommend where to focus retention and pricing efforts.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go
import numpy as np

In [2]:
profiles = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client profiles.xlsx")
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR
0,G1-001,Govt,Yes,NaN,,2020-01-19 00:00:00,More than 200,Finance
1,G1-002,Govt,Yes,NaN,,2020-05-19 00:00:00,More than 200,Security
2,G1-003,Govt,Yes,NaN,,2021-07-18 00:00:00,> 200,Transportation
3,G1-005,Govt,Y,NaN,NaN,2022-02-11 00:00:00,> 200,Security
4,G1-006,Govt,Y,NaN,NaN,2022-03-18 00:00:00,50 ~ 200,Legal


Things to clean:\
    - columns using different values to represent the same info(Sg-based,staff strength)\
    - getting rid of missing values (Country of overseas client)

In [3]:
profiles[['COUNTRY OF OVERSEAS CLIENT','OVERSEAS','SG-BASED']].value_counts()

COUNTRY OF OVERSEAS CLIENT  OVERSEAS  SG-BASED
Malaysia                    Yes                   6
                            Y                     3
India                       Yes                   1
China                       Y                     1
Indonesia                   Yes                   1
Malaysian                   Y                     1
Name: count, dtype: int64

In [4]:
# ---- Clean Client Profiles ----
# Strip stray leading/trailing whitespace on text columns
obj_cols = ['SG-BASED', 'OVERSEAS', 'COUNTRY OF OVERSEAS CLIENT',
            'STAFF STRENGTH', 'SECTOR', 'TYPE', 'CLIENT ID']
for c in obj_cols:
    profiles[c] = profiles[c].apply(lambda v: v.strip() if isinstance(v, str) else v)

# SG-BASED / OVERSEAS: standardise to explicit Y / N ('Yes' and 'Y' -> 'Y', blanks -> 'N')
def yes_no(v):
    return 'Y' if str(v).strip().upper() in ('Y', 'YES') else 'N'
profiles['SG-BASED'] = profiles['SG-BASED'].apply(yes_no)
profiles['OVERSEAS'] = profiles['OVERSEAS'].apply(yes_no)

# COUNTRY OF OVERSEAS CLIENT: unify blanks to NaN, fix 'Malaysian' -> 'Malaysia'
profiles['COUNTRY OF OVERSEAS CLIENT'] = (
    profiles['COUNTRY OF OVERSEAS CLIENT']
    .replace(r'^\s*$', np.nan, regex=True)       # empty / whitespace-only -> NaN
    .replace('Malaysian', 'Malaysia')
)

# Derived country: domestic clients -> 'Singapore', otherwise the overseas country
profiles['country'] = profiles['COUNTRY OF OVERSEAS CLIENT'].fillna('Singapore')

# STAFF STRENGTH: normalise synonym labels, then derive an org_size bucket.
# Metadata rule: 1~49 = Small, 50~200 = Medium, >200 = Large.
# 'Less than 200' is ambiguous (spans Small & Medium) -> kept as 'Unspecified'.
profiles['STAFF STRENGTH'] = profiles['STAFF STRENGTH'].replace(
    {'More than 200': '> 200', 'Less than 49': '1 ~ 49'})
size_map = {'1 ~ 49': 'Small', '50 ~ 200': 'Medium',
            '> 200': 'Large', 'Less than 200': 'Unspecified'}
profiles['org_size'] = profiles['STAFF STRENGTH'].map(size_map)
assert profiles['org_size'].notna().all(), \
    profiles.loc[profiles['org_size'].isna(), 'STAFF STRENGTH'].unique()

# COMMENCEMENT DATE: mix of datetime & string -> uniform datetime64
profiles['COMMENCEMENT DATE'] = pd.to_datetime(profiles['COMMENCEMENT DATE'], errors='coerce')
assert profiles['COMMENCEMENT DATE'].notna().all(), "Unparseable COMMENCEMENT DATE remains"

profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [5]:
profiles['STAFF STRENGTH'].value_counts()

STAFF STRENGTH
> 200            62
50 ~ 200         57
1 ~ 49           22
Less than 200     7
Name: count, dtype: int64

In [6]:
profiles['SG-BASED'].value_counts()

SG-BASED
Y    114
N     34
Name: count, dtype: int64

In [7]:
profiles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   CLIENT ID                   148 non-null    object        
 1   TYPE                        148 non-null    object        
 2   SG-BASED                    148 non-null    object        
 3   OVERSEAS                    148 non-null    object        
 4   COUNTRY OF OVERSEAS CLIENT  34 non-null     object        
 5   COMMENCEMENT DATE           148 non-null    datetime64[ns]
 6   STAFF STRENGTH              148 non-null    object        
 7   SECTOR                      148 non-null    object        
 8   country                     148 non-null    object        
 9   org_size                    148 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 11.7+ KB


In [8]:
survey = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Client survey.xlsx")
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING
0,2021,G1-001,3,4,4,5,8
1,2021,G1-002,3,4,4,2,7
2,2021,G1-003,5,4,3,5,8
3,2021,G1-001,3,4,4,5,8
4,2021,G1-002,3,4,4,2,7


In [9]:
survey['CLIENT ID'].value_counts()

CLIENT ID
G1-001    9
G1-002    9
G1-003    9
G1-006    7
N1-001    5
         ..
P3-027    1
P3-028    1
P3-029    1
P3-030    1
P3-031    1
Name: count, Length: 143, dtype: int64

In [10]:
survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   YEAR                      432 non-null    int64 
 1   CLIENT ID                 432 non-null    object
 2   PRESALES AND PARTNERSHIP  432 non-null    int64 
 3   TECHNICAL EXPERTISE       432 non-null    int64 
 4   PROJECT DELIVERY          432 non-null    int64 
 5   POST-SALES SUPPORT        432 non-null    int64 
 6   NPS RATING                432 non-null    int64 
dtypes: int64(6), object(1)
memory usage: 23.8+ KB


In [11]:
survey.isnull().sum()

YEAR                        0
CLIENT ID                   0
PRESALES AND PARTNERSHIP    0
TECHNICAL EXPERTISE         0
PROJECT DELIVERY            0
POST-SALES SUPPORT          0
NPS RATING                  0
dtype: int64

In [12]:
# ---- Clean Client Survey ----
# Drop 30 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
survey = survey.drop_duplicates()
survey = survey[survey['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived NPS category per respondent (Meta Data thresholds)
def nps_category(score):
    if score >= 9:
        return 'Promoter'    # 9-10
    elif score >= 7:
        return 'Passive'     # 7-8
    return 'Detractor'       # 6 or less
survey['nps_category'] = survey['NPS RATING'].apply(nps_category)

survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [13]:
RC = pd.read_excel("..\AY2627_S1_CA2 datasets & meta data\CA2 datasets & meta data\Revenue and COGS.xlsx")
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER
0,G1-001,2021,148180,17321,25562,41201
1,G1-001,2022,197313,23604,32875,50085
2,G1-001,2023,349899,51332,71695,89089
3,G1-001,2024,268821,40969,56452,76545
4,G1-001,2025,138115,14516,20889,28968


In [14]:
# ---- Clean Revenue and COGS ----
# Drop 21 exact duplicate rows, then drop orphan CLIENT IDs not in Client Profiles
# (P1-010, P1-018, P1-035 -> metadata says such records can be ignored)
RC = RC.drop_duplicates()
RC = RC[RC['CLIENT ID'].isin(profiles['CLIENT ID'])].reset_index(drop=True)

# Derived cost / profitability metrics (formulas from Meta Data)
RC['cogs'] = RC['HARDWARE'] + RC['SOFTWARE'] + RC['MANPOWER']
RC['gross_profit'] = RC['REVENUE'] - RC['cogs']
RC['gross_margin_pct'] = (RC['gross_profit'] / RC['REVENUE'] * 100).round(2)
RC['gross_markup_pct'] = (RC['gross_profit'] / RC['cogs'] * 100).round(2)
RC['loss'] = RC['gross_profit'] < 0   # negative gross profit (all NPO clients) -> flagged, kept

RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [15]:
RC.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392 entries, 0 to 391
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CLIENT ID         392 non-null    object 
 1   YEAR              392 non-null    int64  
 2   REVENUE           392 non-null    int64  
 3   HARDWARE          392 non-null    int64  
 4   SOFTWARE          392 non-null    int64  
 5   MANPOWER          392 non-null    int64  
 6   cogs              392 non-null    int64  
 7   gross_profit      392 non-null    int64  
 8   gross_margin_pct  392 non-null    float64
 9   gross_markup_pct  392 non-null    float64
 10  loss              392 non-null    bool   
dtypes: bool(1), float64(2), int64(7), object(1)
memory usage: 31.1+ KB


In [16]:
RC.isnull().sum()

CLIENT ID           0
YEAR                0
REVENUE             0
HARDWARE            0
SOFTWARE            0
MANPOWER            0
cogs                0
gross_profit        0
gross_margin_pct    0
gross_markup_pct    0
loss                0
dtype: int64

In [17]:
profiles.head()

,CLIENT ID,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,G1-001,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,G1-002,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,G1-003,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,G1-005,Govt,Y,N,NaN,2022-02-11,> 200,Security,Singapore,Large
4,G1-006,Govt,Y,N,NaN,2022-03-18,50 ~ 200,Legal,Singapore,Medium


In [18]:
survey.head()

,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category
0,2021,G1-001,3,4,4,5,8,Passive
1,2021,G1-002,3,4,4,2,7,Passive
2,2021,G1-003,5,4,3,5,8,Passive
3,2021,N1-001,4,4,4,3,8,Passive
4,2021,N1-002,5,3,4,3,7,Passive


In [19]:
RC.head()

,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,loss
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,False
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,False
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,False
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,False
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,False


In [20]:
# ---- Export cleaned datasets ----
import os
os.makedirs('cleaned', exist_ok=True)
profiles.to_csv('cleaned/client_profiles_clean.csv', index=False)
RC.to_csv('cleaned/revenue_cogs_clean.csv', index=False)
survey.to_csv('cleaned/client_survey_clean.csv', index=False)
print('Exported cleaned files:', os.listdir('cleaned'))

Exported cleaned files: ['client_profiles_clean.csv', 'client_survey_clean.csv', 'revenue_cogs_clean.csv']


## DB 1 (Elliot)

#### Graph 1

In [21]:
# ---- Elliot: one base frame for all three charts and the dashboard ----
# RC x profiles x survey, one row per client-year. The survey join is 1:1 on
# (CLIENT ID, YEAR) and complete, so no rows are gained or lost: 392 in, 392 out.
base_df_elliot = (
    RC
    .merge(profiles, on="CLIENT ID", how="left")
    .merge(survey, on=["CLIENT ID", "YEAR"], how="left")
)
assert len(base_df_elliot) == len(RC), "the survey join changed the row count"

# A client's cohort is the year we signed them, and it never changes.
base_df_elliot["cohort"] = base_df_elliot["COMMENCEMENT DATE"].dt.year
base_df_elliot["cohort_label"] = "Joined " + base_df_elliot["cohort"].astype(str)
base_df_elliot["tenure"] = base_df_elliot["YEAR"] - base_df_elliot["cohort"]   # 0 = first year with us

cohort_order_elliot = [f"Joined {year}" for year in sorted(base_df_elliot["cohort"].unique())]

# Cool -> warm rather than 7 shades of one colour, so neighbouring lines stay tellable apart.
cohort_colours_elliot = dict(zip(cohort_order_elliot,
                                 ["#08306b", "#1f6fb4", "#2ca7a4", "#6bbf59",
                                  "#e8b73a", "#e06c39", "#b23a5f"]))
average_label_elliot = "Company average"

# ---- Sector axis labels (Graph 3) ----
# Ordering and the n= counts are computed once on the FULL dataset so the x-axis
# stays put when the dashboard filters change - a category order that reshuffles
# under every filter is unreadable.
sector_order_elliot = (base_df_elliot.groupby("SECTOR")["gross_margin_pct"].median()
                       .sort_values().index.tolist())
n_by_sector_elliot = base_df_elliot["SECTOR"].value_counts()
sector_labels_elliot = {s: f"{s}<br>(n={n_by_sector_elliot[s]})" for s in sector_order_elliot}
sector_label_order_elliot = [sector_labels_elliot[s] for s in sector_order_elliot]

base_df_elliot["sector_label"] = base_df_elliot["SECTOR"].map(sector_labels_elliot)
base_df_elliot["point_text"] = base_df_elliot["CLIENT ID"] + " · " + base_df_elliot["YEAR"].astype(str)

base_df_elliot.head()


,CLIENT ID,YEAR,REVENUE,HARDWARE,SOFTWARE,MANPOWER,cogs,gross_profit,gross_margin_pct,gross_markup_pct,...,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category,cohort,cohort_label,tenure,sector_label,point_text
0,G1-001,2021,148180,17321,25562,41201,84084,64096,43.26,76.23,...,4,4,5,8,Passive,2020,Joined 2020,1,Finance<br>(n=5),G1-001 · 2021
1,G1-001,2022,197313,23604,32875,50085,106564,90749,45.99,85.16,...,5,3,4,9,Promoter,2020,Joined 2020,2,Finance<br>(n=5),G1-001 · 2022
2,G1-001,2023,349899,51332,71695,89089,212116,137783,39.38,64.96,...,4,3,4,7,Passive,2020,Joined 2020,3,Finance<br>(n=5),G1-001 · 2023
3,G1-001,2024,268821,40969,56452,76545,173966,94855,35.29,54.53,...,5,3,4,8,Passive,2020,Joined 2020,4,Finance<br>(n=5),G1-001 · 2024
4,G1-001,2025,138115,14516,20889,28968,64373,73742,53.39,114.55,...,4,5,5,10,Promoter,2020,Joined 2020,5,Finance<br>(n=5),G1-001 · 2025


In [22]:
def blank_figure(message):
    """Empty chart with a message on it, for when a filter selects no rows."""
    fig = go.Figure()
    fig.update_layout(
        height=420,
        xaxis={"visible": False},
        yaxis={"visible": False},
        annotations=[{"text": message, "showarrow": False,
                      "xref": "paper", "yref": "paper", "x": 0.5, "y": 0.5,
                      "font": {"size": 15, "color": "#888888"}}],
    )
    return fig


def cohort_curves_elliot(data):
    """Profit per intake per year, plus a company average row per year.

    This is a function and not a saved table because the dashboard has to
    recalculate it every time a filter changes.
    """
    curves = (
        data.groupby(["cohort_label", "YEAR"])
        .agg(total_profit=("gross_profit", "sum"),
             revenue=("REVENUE", "sum"),
             clients=("CLIENT ID", "nunique"),
             tenure=("tenure", "first"))
        .reset_index()
        .sort_values(["cohort_label", "YEAR"])
    )
    # Per client, not per intake. The 2024 intake beats the 2019 intake on total
    # dollars only because it has 40 clients against 9, so dividing by headcount
    # is what makes the lines comparable.
    curves["profit_per_client"] = curves["total_profit"] / curves["clients"]
    curves["revenue_per_client"] = curves["revenue"] / curves["clients"]

    average = (
        data.groupby("YEAR")
        .agg(total_profit=("gross_profit", "sum"),
             revenue=("REVENUE", "sum"),
             clients=("CLIENT ID", "nunique"))
        .reset_index()
    )
    average["profit_per_client"] = average["total_profit"] / average["clients"]
    average["revenue_per_client"] = average["revenue"] / average["clients"]
    # The benchmark's "total" is the average of the intake totals, not their sum.
    # Summing would give the whole company and flatten every other line.
    average["mean_cohort_total"] = (curves.groupby("YEAR")["total_profit"].mean()
                                    .reindex(average["YEAR"]).values)
    average["tenure"] = -1
    average["cohort_label"] = average_label_elliot
    return curves, average


curve_df_elliot, average_df_elliot = cohort_curves_elliot(base_df_elliot)

# How much of each year's profit came from clients we already had.
existing_share_elliot = (
    base_df_elliot[base_df_elliot["cohort"] < base_df_elliot["YEAR"]]
    .groupby("YEAR")["gross_profit"].sum()
    / base_df_elliot.groupby("YEAR")["gross_profit"].sum() * 100
)

curve_df_elliot.pivot(index="cohort_label", columns="YEAR", values="profit_per_client").round(0)


YEAR,2021,2022,2023,2024,2025
cohort_label,,,,,
Joined 2019,89434.0,87274.0,117709.0,175806.0,269324.0
Joined 2020,126041.0,127603.0,182976.0,179831.0,332868.0
Joined 2021,117021.0,70054.0,88129.0,202358.0,273628.0
Joined 2022,NaN,136965.0,153778.0,136290.0,222425.0
Joined 2023,NaN,NaN,93407.0,151746.0,216160.0
Joined 2024,NaN,NaN,NaN,122980.0,124612.0
Joined 2025,NaN,NaN,NaN,NaN,154779.0


In [23]:
# The three views this chart can show. "Revenue per client" is worth having:
# margin only varies 3.7pp across all seven intakes, so that curve comes out
# almost the same shape as profit per client, which shows the tenure effect is
# clients buying more rather than clients being charged more.
g1_views_elliot = {
    "Gross profit per client":     {"col": "profit_per_client",
                                    "avg_col": "profit_per_client",
                                    "axis": "Gross profit per client ($)"},
    "Revenue per client":          {"col": "revenue_per_client",
                                    "avg_col": "revenue_per_client",
                                    "axis": "Revenue per client ($)"},
    "Total profit for the intake": {"col": "total_profit",
                                    "avg_col": "mean_cohort_total",
                                    "axis": "Total gross profit for the intake ($)"},
}

# Every trace carries the same custom_data in this order. The hover text reads
# from custom_data and never from %{y}, because y means something different in
# each view and a %{y} label would say "per client" over an intake total.
g1_custom_elliot = ["tenure", "clients", "total_profit",
                    "profit_per_client", "revenue_per_client"]

g1_hover_elliot = (
    "<b>%{fullData.name}</b><br>%{x} | year %{customdata[0]} of the relationship"
    "<br>$%{customdata[3]:,.0f} gross profit per client"
    "<br>$%{customdata[4]:,.0f} revenue per client"
    "<br>$%{customdata[2]:,.0f} in total across %{customdata[1]} clients<extra></extra>"
)


def fig_cohort_elliot(data, view="Gross profit per client", show_buttons=False):
    """Line chart: profit per client by the year each client was signed.

    show_buttons=True adds the buttons inside the chart, which is what the
    notebook version uses. The dashboard leaves them off and uses a radio
    button instead, so the same function serves both.
    """
    if data.empty:
        return blank_figure("No clients match these filters")

    curves, average = cohort_curves_elliot(data)
    view_info = g1_views_elliot[view]

    # With a narrow filter there may not be enough clients in each intake to
    # draw a meaningful line. Finance, for example, has one client a year, which
    # would come out as seven single dots.
    latest_year = curves["YEAR"].max()
    big_enough = curves[(curves["YEAR"] == latest_year) & (curves["clients"] >= 3)]
    if len(big_enough) < 3:
        return blank_figure("Too few clients per intake for this selection")

    fig = px.line(
        curves,
        x="YEAR",
        y=view_info["col"],
        color="cohort_label",                     # one line per intake year
        color_discrete_map=cohort_colours_elliot,
        category_orders={"cohort_label": cohort_order_elliot},
        markers=True,                             # data is yearly, so show points
        custom_data=g1_custom_elliot,
        labels={"YEAR": "Year", "cohort_label": "Client intake"},
    )
    fig.update_traces(hovertemplate=g1_hover_elliot)

    # The company average, added as one more line.
    average_fig = px.line(average, x="YEAR", y=view_info["avg_col"], markers=True,
                          custom_data=g1_custom_elliot,
                          color_discrete_sequence=["#111111"])
    average_fig.update_traces(
        name=average_label_elliot,
        showlegend=True,
        line={"width": 4, "dash": "dot"},
        marker={"size": 10},
        hovertemplate="<b>Company average</b><br>%{x}"
                      "<br>$%{customdata[3]:,.0f} gross profit per client"
                      " across %{customdata[1]} clients"
                      "<br>$%{customdata[4]:,.0f} revenue per client"
                      "<br>$%{customdata[2]:,.0f} average per intake<extra></extra>")
    fig.add_traces(average_fig.data)

    top = max(max(trace.y) for trace in fig.data) * 1.15
    fig.update_layout(
        title={"text": "Does more profit come from longer-term clients?",
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        xaxis={"title": "Year", "tickmode": "linear", "dtick": 1},
        yaxis={"title": view_info["axis"], "tickprefix": "$",
               "tickformat": "~s", "range": [0, top]},
        legend={"title": "Client intake"},
        plot_bgcolor="white",
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        # The dashboard version has no buttons above the title, so it can
        # give the top margin back to the plot itself.
        margin={"t": 140 if show_buttons else 46, "b": 80},
    )

    if show_buttons:
        # One list of y values per view, in the same order as fig.data, so a
        # button can swap all the lines at once.
        buttons = []
        for name, info in g1_views_elliot.items():
            y_values = []
            for trace in fig.data:
                if trace.name == average_label_elliot:
                    y_values.append(list(average[info["avg_col"]]))
                else:
                    one_intake = curves[curves["cohort_label"] == trace.name]
                    y_values.append(list(one_intake[info["col"]]))
            buttons.append({
                "label": name,
                "method": "update",
                "args": [{"y": y_values},
                         {"yaxis.title.text": info["axis"],
                          "yaxis.range": [0, max(map(max, y_values)) * 1.15]}],
            })
        fig.update_layout(
            width = 1100,
            updatemenus=[{"type": "buttons", "direction": "right",
                          "buttons": buttons, "x": 0.5, "xanchor": "center",
                          "y": 1.10, "yanchor": "top", "showactive": True}])

    return fig


fig_cohort_elliot(base_df_elliot, show_buttons=True).show()


### Insights

1. **Yes. Longer term clients earn us more profit.** In 2025, a client we signed in 2019, 2020 or 2021 earned us \\$269k to \\$333k in gross profit. A client we signed in 2024 earned us \\$125k. That is about twice as much, and up to 2.5 times for the best group. The company average is \\$195k. Five of the seven joining years sit above it and only the two newest sit below.

2. **They give us more work. They do not pay higher prices.** Switch to *Revenue per client*. A client from 2019 to 2021 brings in \\$679k a year. A client from 2024 or 2025 brings in \\$309k. Both sit at almost the same gross margin, 44.5% against 43.9%. We sell projects, not subscriptions. So our older clients keep coming back with more projects, or with bigger ones. It is also not one lucky group. Every line goes up. The 2019 group went from \\$89k to \\$269k per client. We have lost only 7 clients in five years, so these are the same accounts growing.

3. **Repeat work is what kept profit growing, not new clients.** Switch to *Total profit for the intake*. In 2025 the 2023 group earned us \\$6.5M from 30 clients. The 2024 group earned \\$5.0M from 40 clients. Newer groups bring more clients. Older groups bring more money per client. In 2025, 86% of our gross profit came from clients we signed in earlier years. Profit still rose 54%, from \\$16.8M to \\$25.9M, even though we signed 26 new clients instead of 44.


#### Graph 2

In [24]:
cogs_components_elliot = ["HARDWARE", "SOFTWARE", "MANPOWER"]   # dataset field names
component_colours_elliot = {"HARDWARE": "#8c6d31", "SOFTWARE": "#6b6ecf", "MANPOWER": "#e6550d"}


In [25]:
def yearly_costs_elliot(data):
    """Yearly cost totals, margin, each cost as a share of COGS, and per client."""
    yearly = (
        data.groupby("YEAR")
        .agg(HARDWARE=("HARDWARE", "sum"),
             SOFTWARE=("SOFTWARE", "sum"),
             MANPOWER=("MANPOWER", "sum"),
             REVENUE=("REVENUE", "sum"),
             gross_profit=("gross_profit", "sum"),
             clients=("CLIENT ID", "nunique"))
        .reset_index()
    )
    yearly["margin"] = yearly["gross_profit"] / yearly["REVENUE"] * 100
    cogs_total = yearly[cogs_components_elliot].sum(axis=1)
    for component in cogs_components_elliot:
        yearly[component + " share"] = yearly[component] / cogs_total * 100
        yearly[component + " per client"] = yearly[component] / yearly["clients"]
    return yearly


yearly_costs_elliot(base_df_elliot).round(1)


,YEAR,HARDWARE,SOFTWARE,MANPOWER,REVENUE,gross_profit,clients,margin,HARDWARE share,HARDWARE per client,SOFTWARE share,SOFTWARE per client,MANPOWER share,MANPOWER per client
0,2021,982459,1592882,1771571,7581911,3234999,29,42.7,22.6,33877.9,36.6,54927.0,40.8,61088.7
1,2022,1128725,1987706,2370926,10170124,4682767,42,46.0,20.6,26874.4,36.2,47326.3,43.2,56450.6
2,2023,2400758,4116084,4880118,20469640,9072680,74,44.3,21.1,32442.7,36.1,55622.8,42.8,65947.5
3,2024,4546884,7599530,9114745,38038669,16777510,114,44.1,21.4,39884.9,35.7,66662.5,42.9,79953.9
4,2025,6775006,11658495,14466160,58820894,25921233,133,44.1,20.6,50939.9,35.4,87657.9,44.0,108768.1


In [26]:
# Three ways of reading the same cost stack. "Per client $" is the one that
# links this chart to Graph 1: manpower per client went up 78% ($61k to $109k)
# against 75% growth in profit per client, which is why margin stays near 44%.
g2_modes_elliot = {
    "Total $":         {"suffix": "", "axis": "Cost of Goods Sold ($)",
                        "tickprefix": "$", "tickformat": "~s"},
    "% share of COGS": {"suffix": " share", "axis": "Share of COGS (%)",
                        "tickprefix": "", "tickformat": ".0f"},
    "Per client $":    {"suffix": " per client", "axis": "COGS per client ($)",
                        "tickprefix": "$", "tickformat": "~s"},
}


def fig_cogs_elliot(data, mode="Total $", show_buttons=False):
    """Stacked bars of the three cost types, with gross margin on a second axis.

    The client type dropdown this chart used to have is gone. In the dashboard
    that filter sits in the navigation bar and controls all three charts, so
    having a second copy here would just be confusing.
    """
    if data.empty:
        return blank_figure("No clients match these filters")

    yearly = yearly_costs_elliot(data)
    mode_info = g2_modes_elliot[mode]
    fig = go.Figure()

    for component in cogs_components_elliot:
        # All three numbers go in customdata so the hover text stays right in
        # every mode. Using %{y} would label two of the three modes wrongly.
        fig.add_trace(go.Bar(
            x=yearly["YEAR"],
            y=yearly[component + mode_info["suffix"]],
            name=component.title(),
            marker_color=component_colours_elliot[component],
            customdata=np.stack([yearly[component],
                                 yearly[component + " share"],
                                 yearly[component + " per client"]], axis=-1),
            hovertemplate=component.title()
                          + ": $%{customdata[0]:,.0f}"
                            " | %{customdata[1]:.0f}% of COGS"
                            " | $%{customdata[2]:,.0f} per client<extra></extra>",
        ))

    # Margin is a percentage, so it looks the same in all three modes.
    fig.add_trace(go.Scatter(
        x=yearly["YEAR"],
        y=yearly["margin"],
        name="Gross Margin %",
        mode="lines+markers+text",
        text=yearly["margin"].round(1).astype(str) + "%",
        textposition="top center",
        line={"color": "#222222", "width": 3},
        marker={"size": 9},
        yaxis="y2",                              # second axis, percent not dollars
        hovertemplate="Margin: %{y:.1f}%<extra></extra>",
    ))

    fig.update_layout(
        barmode="stack",
        title={"text": "COGS composition and gross margin: " + mode,
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        xaxis={"title": "Year", "tickmode": "linear", "dtick": 1},
        yaxis={"title": mode_info["axis"], "tickprefix": mode_info["tickprefix"],
               "tickformat": mode_info["tickformat"]},
        yaxis2={"title": "Gross Margin (%)", "overlaying": "y", "side": "right",
                "range": [0, 60], "showgrid": False},
        hovermode="x",
        legend={"orientation": "h", "y": -0.18},
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        # The dashboard version has no buttons above the title, so it can
        # give the top margin back to the plot itself.
        margin={"t": 140 if show_buttons else 46, "b": 80},
    )

    if show_buttons:
        buttons = []
        for name, info in g2_modes_elliot.items():
            y_values = [list(yearly[component + info["suffix"]])
                        for component in cogs_components_elliot]
            y_values.append(list(yearly["margin"]))       # margin line is unchanged
            buttons.append({
                "label": name,
                "method": "update",
                "args": [{"y": y_values},
                         {"yaxis.title.text": info["axis"],
                          "yaxis.tickprefix": info["tickprefix"],
                          "yaxis.tickformat": info["tickformat"],
                          "title.text": "COGS composition and gross margin: " + name}],
            })
        fig.update_layout(
            width = 1100,
            updatemenus=[{"type": "buttons", "direction": "right",
                          "buttons": buttons, "x": 0.5, "xanchor": "center",
                          "y": 1.10, "yanchor": "top", "showactive": True}])

    return fig


fig_cogs_elliot(base_df_elliot, show_buttons=True).show()


### Insights

1. **Where each dollar of a project goes.** *Switch to % share of COGS*. Manpower is our biggest cost at 44% of what we spend. Software licences are 35% and hardware is 21%. For every \\$100 a client paid us in 2025, about \\$25 goes on our own delivery work. About \\$31 goes on hardware and software that we buy in and pass on to the client. The last \\$44 is gross profit. So about a third of the price is bought-in hardware and licences and about a quarter is the work we do ourselves.

2. **Profit Margin has not decreased even though business grew almost 8 times.** Revenue went from \\$7.6M in 2021 to \\$58.8M in 2025. Gross margin stayed between 42.7% and 46.0% in every single year. In 2021 we kept \\$42.67 out of every \\$100 a client paid us. In 2025 we kept \\$44.07. We did not win that growth by dropping our prices, and the shape of the business did not change as it grew.

3. **Increasing cost of manpower.** Switch to *Per client $*. Manpower cost per client is up 78% since 2021, from \\$61k to \\$109k. Revenue per client is up 69%, so manpower is growing faster than the money coming in. Total manpower spend has grown 8.2 times, against 7.8 times for revenue and only 4.6 times for client numbers. Doing more of the work ourselves does not earn a better margin (the link is only +0.06 once charities are left out), so the risk is capacity, not profit. We can buy more hardware in a week. We cannot hire experienced integrators that fast.


### Graph 3

In [27]:
# refline is True or False rather than a fixed number, because the reference
# line is worked out from whatever rows are on screen at the time.
metrics_elliot = {
    # dtick is the gap between y axis labels. Left to itself the margin axis
    # only draws 0 and 50 in the small dashboard version, which is too few to
    # read the boxes against, so we ask for one every 20 points.
    "Gross Margin %": {"col": "gross_margin_pct", "axis_type": "linear",
                       "fmt": ".1f", "suffix": "%", "refline": True,
                       "dtick": 20},
    "Revenue":        {"col": "REVENUE", "axis_type": "log",   # revenue is very skewed
                       "fmt": ",.0f", "suffix": " $", "refline": False,
                       "dtick": None},
}
default_metric_elliot = "Gross Margin %"


def ref_shapes_elliot(metric, data):
    """Dotted line at the overall margin for the rows being shown."""
    if not metric["refline"] or data.empty:
        return []
    overall = data["gross_profit"].sum() / data["REVENUE"].sum() * 100
    return [{"type": "line", "xref": "paper", "x0": 0, "x1": 1,
             "y0": overall, "y1": overall,
             "line": {"dash": "dot", "color": "#555555", "width": 2}}]


In [28]:
# Charity gets its own trace so it can be coloured differently. It is the one
# sector whose whole distribution sits below the others.
sector_groups_elliot = [("Other sectors", "#7f8fa6", False),
                        ("Charity", "#d62728", True)]


def fig_sector_elliot(data, metric_name="Gross Margin %", show_buttons=False):
    """Box plot of one metric across the sectors, with Charity picked out."""
    if data.empty:
        return blank_figure("No clients match these filters")

    # With buttons we build both metrics and hide one. Without, just the one asked for.
    metrics_to_build = list(metrics_elliot) if show_buttons else [metric_name]

    is_charity = data["SECTOR"] == "Charity"
    fig = go.Figure()
    for name in metrics_to_build:
        metric = metrics_elliot[name]
        for group_name, group_colour, charity_group in sector_groups_elliot:
            subset = data[is_charity] if charity_group else data[~is_charity]
            fig.add_trace(go.Box(
                x=subset["sector_label"],
                y=subset[metric["col"]],
                name=group_name,
                marker_color=group_colour,
                boxpoints="all", jitter=0.5, pointpos=0,   # points on the box
                marker={"size": 4, "opacity": 0.5},
                line={"width": 1.5},
                visible=(name == metric_name),
                text=subset["point_text"],
                hovertemplate="%{text}<br>%{y:" + metric["fmt"] + "}"
                              + metric["suffix"] + "<extra></extra>",
            ))

    fig.update_layout(
        title={"text": "Distribution of " + metric_name + " by sector",
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        xaxis={"title": "Sector (ordered by median margin, worst to best)",
               "categoryorder": "array", "categoryarray": sector_label_order_elliot},
        yaxis={"title": metric_name,
               "type": metrics_elliot[metric_name]["axis_type"],
               "dtick": metrics_elliot[metric_name]["dtick"]},
        shapes=ref_shapes_elliot(metrics_elliot[metric_name], data),
        boxmode="group",
        legend={"orientation": "h", "y": -0.25},
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        # The dashboard version has no buttons above the title, so it can
        # give the top margin back to the plot itself.
        margin={"t": 140 if show_buttons else 46, "b": 80},
    )

    if show_buttons:
        buttons = []
        for name in metrics_elliot:
            # Two traces per metric, so show that metric's pair and hide the rest.
            visible = []
            for built in metrics_to_build:
                for _ in sector_groups_elliot:
                    visible.append(built == name)
            buttons.append({
                "label": name,
                "method": "update",
                "args": [{"visible": visible},
                         {"yaxis.title.text": name,
                          "yaxis.type": metrics_elliot[name]["axis_type"],
                          "yaxis.dtick": metrics_elliot[name]["dtick"],
                          "shapes": ref_shapes_elliot(metrics_elliot[name], data),
                          "title.text": "Distribution of " + name + " by sector"}],
            })
        fig.update_layout(
            width = 1100,
            updatemenus=[{"type": "buttons", "direction": "right",
                          "buttons": buttons, "x": 0.5, "xanchor": "center",
                          "y": 1.10, "yanchor": "top", "showactive": True}])

    return fig


fig_sector_elliot(base_df_elliot, show_buttons=True).show()


### Insights

1. **Charity work earns far less than anything else we do.** The whole charity box sits below every other sector. Its median gross margin is 18.6%, while every other sector is between 41.8% and 47.3%. Put another way, we add about 17% on top of what a charity job costs us. For private and government clients we add about 80%. The meta data says this mark-up is what pays for sales, admin and the other overheads that are not in this dataset. Charity work almost certainly does not cover its share of them.

2. **In money terms it is tiny.** Switch to *Revenue*. Charity is about \\$1.0M out of \\$135M over the five years. That is under 1% of everything we charge. So the amount at stake is small. This is a choice we should make on purpose, not a hole in the accounts. We can keep serving these clients cheaply as a deliberate contribution, or we can start pricing the work properly.

3. **Every other sector is very consistent.** Leave charity aside and the nine remaining sectors all sit within about 5 percentage points of each other, from 41.8% to 47.3%. There is no sector we should avoid and no sector we should chase on profit grounds. Margin is not a useful way to pick which clients to go after. (Worth pairing with Zafran's Graph 1. Charity has the highest share of promoters of any sector at 62.5%, and almost no detractors at 4.2%. The clients who rate us best are the ones we earn the least from.)


## DB 2 (Zafran)

#### Graph 1

NPS category mix sunburst

In [29]:
# ---- Zafran: one base frame for all three charts and the dashboard ----
# Client survey joined to client profiles, one row per client-year. The join is
# many-to-one on CLIENT ID and complete, so no rows are gained or lost.
base_df_zafran = survey.merge(profiles, on="CLIENT ID", how="inner")
assert len(base_df_zafran) == len(survey), "the profiles join changed the row count"

# Fixed colour per NPS category, keyed by the category name and never a list
# read by position. The dashboard filters change which category is the most
# common, and a positional map would then hand Promoter the Detractor colour.
nps_colours_zafran = {
    "Promoter": "#00CC96",     # green
    "Passive": "#636EFA",      # blue
    "Detractor": "#EF553B",    # red
}
nps_order_zafran = ["Promoter", "Passive", "Detractor"]

# One colour per sector for the sunburst inner ring, worked out once on the FULL
# dataset so a sector keeps the same colour when a filter takes it away and puts
# it back again.
sectors_all_zafran = sorted(base_df_zafran["SECTOR"].dropna().unique())
sector_palette_zafran = px.colors.qualitative.Set3
sector_colours_zafran = {
    sector: sector_palette_zafran[i % len(sector_palette_zafran)]
    for i, sector in enumerate(sectors_all_zafran)
}

# The survey scores this member works with. NPS RATING is the headline question
# and the other four are the sub-scores behind it.
metrics_zafran = {
    "NPS Rating": "NPS RATING",
    "Presales & Partnership": "PRESALES AND PARTNERSHIP",
    "Technical Expertise": "TECHNICAL EXPERTISE",
    "Project Delivery": "PROJECT DELIVERY",
    "Post-Sales Support": "POST-SALES SUPPORT",
}
subscores_zafran = [col for label, col in metrics_zafran.items() if label != "NPS Rating"]

metric_colours_zafran = {
    "NPS Rating": "#000000",
    "Presales & Partnership": "#1f77b4",
    "Technical Expertise": "#ff7f0e",
    "Project Delivery": "#2ca02c",
    "Post-Sales Support": "#d62728",
}

base_df_zafran.head()


,YEAR,CLIENT ID,PRESALES AND PARTNERSHIP,TECHNICAL EXPERTISE,PROJECT DELIVERY,POST-SALES SUPPORT,NPS RATING,nps_category,TYPE,SG-BASED,OVERSEAS,COUNTRY OF OVERSEAS CLIENT,COMMENCEMENT DATE,STAFF STRENGTH,SECTOR,country,org_size
0,2021,G1-001,3,4,4,5,8,Passive,Govt,Y,N,NaN,2020-01-19,> 200,Finance,Singapore,Large
1,2021,G1-002,3,4,4,2,7,Passive,Govt,Y,N,NaN,2020-05-19,> 200,Security,Singapore,Large
2,2021,G1-003,5,4,3,5,8,Passive,Govt,Y,N,NaN,2021-07-18,> 200,Transportation,Singapore,Large
3,2021,N1-001,4,4,4,3,8,Passive,NPO,Y,N,NaN,2019-06-29,1 ~ 49,Charity,Singapore,Small
4,2021,N1-002,5,3,4,3,7,Passive,NPO,Y,N,NaN,2021-05-18,1 ~ 49,Charity,Singapore,Small


In [30]:
"""
Plotly Express - Promoter / Passive / Detractor mix by sector (sunburst).

Interactive element: a year slider with an "All years" step.

Two rings, two different colour schemes:
    inner ring (SECTOR)       -> one colour per sector
    outer ring (nps_category) -> the Promoter / Passive / Detractor colours

px.sunburst() takes a single `color` column, so it cannot colour two levels from
two different maps on its own. The way round it is to build the figure with
px.sunburst() as normal, coloured by nps_category, then overwrite marker.colors
afterwards: a node whose label is a sector name takes its sector colour, and
every other node keeps the category colour px gave it.

px.sunburst() also has no animation_frame, so the year slider is built by
working out each year's node arrays up front and swapping them in with restyle
steps.
"""


def sunburst_zafran(data):
    """The px.sunburst for one slice of the data, with both colour schemes on it.

    Pulled out into its own function because the year slider needs the same
    figure built once per year.
    """
    counts = data.groupby(["SECTOR", "nps_category"]).size().reset_index(name="count")

    fig = px.sunburst(
        counts,
        path=["SECTOR", "nps_category"],       # inner ring, then outer ring
        values="count",
        color="nps_category",
        color_discrete_map=nps_colours_zafran,
    )
    # A node whose label is a sector name is on the inner ring. Everything else
    # is a Promoter / Passive / Detractor leaf and keeps the colour px gave it.
    fig.update_traces(marker={"colors": [
        sector_colours_zafran.get(label, nps_colours_zafran.get(label, "#CCCCCC"))
        for label in fig.data[0].labels
    ]})
    return fig


def fig_nps_mix_zafran(data, show_buttons=False):
    """Sunburst: how each sector's clients split into Promoters, Passives and Detractors.

    show_buttons=True adds the year slider inside the chart, which is what the
    notebook version uses. The dashboard leaves it off and drives the year from
    the filter bar instead, so one function serves both.
    """
    if data.empty:
        return blank_figure("No clients match these filters")

    fig = sunburst_zafran(data)

    # percentRoot rather than percentParent, because a sector node has no parent
    # and would read as a share of nothing.
    fig.update_traces(
        hovertemplate="<b>%{label}</b><br>%{value} client responses"
                      "<br>%{percentRoot:.1%} of the selection<extra></extra>")

    fig.update_layout(
        title={"text": "How does each sector split into Promoters and Detractors?",
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        # An explicit width for the notebook version. Left unset for the
        # dashboard so the chart fills whatever width its panel gives it.
        autosize=True,
        height=700 if show_buttons else DASH_CHART_HEIGHT + 40,
        # The year slider is drawn in the bottom margin rather than inside the
        # plot, so the notebook version has to reserve the room for it. At 80px
        # the year labels ran off the bottom of the canvas.
        margin={"t": 60, "b": 150 if show_buttons else 0},
    )

    if show_buttons:
        # One set of node arrays per year, so a slider step can swap the whole
        # ring in place.
        steps = []
        for option in ["All years"] + [str(y) for y in sorted(data["YEAR"].unique())]:
            subset = data if option == "All years" else data[data["YEAR"] == int(option)]
            trace = sunburst_zafran(subset).data[0]
            steps.append({
                "method": "restyle",
                "label": option,
                "args": [{"ids": [trace.ids], "labels": [trace.labels],
                          "parents": [trace.parents], "values": [trace.values],
                          "marker": [{"colors": list(trace.marker.colors)}]}],
            })
        fig.update_layout(
            sliders=[{"active": 0, "currentvalue": {"prefix": "Year: "},
                      "pad": {"t": 30}, "steps": steps}])

    return fig


fig_nps_mix_zafran(base_df_zafran, show_buttons=True).show()


### Insights

1. Detractors made up only about 7.5% of clients in 2025, the lowest share across the five-year period. This suggests client satisfaction improved in the latest year despite the growing client base.

2. Manufacturing remained one of the main sources of dissatisfied clients, but its detractor count fell from 5 in 2024 to 3 in 2025, a 40% reduction. The changes between these two years are therefore worth investigating to identify what improved.

#### Graph 2

NPS category mix by year, as raw counts or as a 100% stacked share.


In [31]:
# Count of clients in each NPS category per year. This is the table the stacked
# bar chart below is drawn from.
nps_by_year_zafran = (
    base_df_zafran.groupby("YEAR")["nps_category"].value_counts().reset_index(name="count")
)
nps_by_year_zafran


,YEAR,nps_category,count
0,2021,Passive,19
1,2021,Detractor,5
2,2021,Promoter,5
3,2022,Passive,22
4,2022,Promoter,14
5,2022,Detractor,6
6,2023,Passive,44
7,2023,Promoter,24
8,2023,Detractor,6
9,2024,Promoter,52


In [32]:
# Two ways of reading the same stack. Raw counts show the client base growing,
# 100% stacked takes the growth out and shows the mix moving towards Promoters,
# which is the point of the chart.
g2_modes_zafran = {
    "Raw counts": {"barnorm": None, "axis": "Number of clients"},
    "100% stacked": {"barnorm": "percent", "axis": "Share of clients (%)"},
}


def fig_nps_stack_zafran(data, mode="Raw counts", show_buttons=False):
    """Stacked bars: the Promoter / Passive / Detractor split of every year.

    show_buttons=True puts the two mode buttons inside the chart for the
    notebook version. The dashboard leaves them off and drives the same argument
    from a radio button, so the chart never carries two sets of controls.
    """
    if data.empty:
        return blank_figure("No clients match these filters")

    counts = data.groupby(["YEAR", "nps_category"]).size().reset_index(name="count")
    totals = counts.groupby("YEAR")["count"].sum()
    mode_info = g2_modes_zafran[mode]

    fig = go.Figure()
    # Looping over the fixed order rather than over whatever the data contains
    # keeps the stack in the same order, and the colours attached to the right
    # category, under every filter.
    for category in nps_order_zafran:
        one = counts[counts["nps_category"] == category].copy()
        if one.empty:
            continue
        one["total"] = one["YEAR"].map(totals)
        one["share"] = one["count"] / one["total"] * 100
        fig.add_trace(go.Bar(
            x=one["YEAR"],
            y=one["count"],
            name=category,
            marker_color=nps_colours_zafran[category],
            # The count and the share both travel in customdata, so the hover
            # text stays right in whichever mode is showing. A %{y} label would
            # read as a headcount in one mode and a percentage in the other.
            customdata=one[["count", "share", "total"]].values,
            hovertemplate=(category + ": %{customdata[0]} of %{customdata[2]} clients"
                           " (%{customdata[1]:.1f}%)<extra></extra>"),
        ))

    fig.update_layout(
        barmode="stack",
        barnorm=mode_info["barnorm"],
        hovermode="x unified",
        title={"text": "Is the client mix shifting towards Promoters?",
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        xaxis={"title": "Year", "tickmode": "linear", "dtick": 1},
        yaxis={"title": mode_info["axis"]},
        legend={"title": "NPS category"},
        plot_bgcolor="white",
        # An explicit width for the notebook version. Without one the chart is
        # drawn at whatever width the output area happens to report first, and
        # only snaps to the full width once an interaction forces a redraw. The
        # dashboard leaves it unset so the chart fills its panel.
        autosize=True,
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        margin={"t": 140 if show_buttons else 46, "b": 60},
    )

    if show_buttons:
        # relayout rather than restyle: the bars do not change, only how plotly
        # normalises them. An empty string is how barnorm is switched off.
        buttons = [{"label": name,
                    "method": "relayout",
                    "args": [{"barnorm": info["barnorm"] or "",
                              "yaxis.title.text": info["axis"]}]}
                   for name, info in g2_modes_zafran.items()]
        fig.update_layout(
            updatemenus=[{"type": "buttons", "direction": "right", "buttons": buttons,
                          "x": 0.5, "xanchor": "center", "y": 1.10, "yanchor": "top",
                          "showactive": True}])

    return fig


fig_nps_stack_zafran(base_df_zafran, show_buttons=True).show()


### Insights

1. Client satisfaction improved as the client base grew. By 2025, Promoters made up approximately 52.6% of clients, exceeding half of the total client base.

2. Detractors fell to approximately 7.5% of clients in 2025, the lowest proportion across the five-year period.

3. 2024 stands out as an anomaly, with Detractors making up approximately 16.7% of clients compared with 8.1% in 2023 and 7.5% in 2025.

#### Graph 3

NPS Rating and sub score trend line chart

In [33]:
"""
Plotly Graph Objects - NPS rating and the four sub-scores, year by year.

Interactive elements: a range slider under the x axis, and click any legend
entry to drop that line out of the comparison.

The four sub-scores are answered on a 1 to 5 scale and the NPS rating on 0 to
10, so the sub-scores are doubled to put every line on one comparable axis.
That doubling happens on a local copy inside the function. Rebinding the global
`survey` frame here would leave every later cell, and every dashboard callback,
reading scores at twice their real size, and would double them again on every
re-run of the cell.
"""


def fig_nps_trend_zafran(data, show_buttons=False):
    """Line chart: the NPS rating and its four sub-scores over the years.

    show_buttons=True adds the range slider under the x axis for the notebook
    version. The dashboard leaves it off, because the year range on the filter
    bar already does that job and the slider would only cost chart height.
    """
    if data.empty:
        return blank_figure("No clients match these filters")
    if len(data) < 3:
        return blank_figure("Too few survey responses for this selection")

    scaled = data.copy()
    scaled[subscores_zafran] = scaled[subscores_zafran] * 2

    # Correlation with the headline NPS rating, worked out on whatever rows are
    # on screen so the hover text stays true when the dashboard filters.
    # Doubling a column does not change its correlation, so the scaled copy
    # gives the same answer as the raw one.
    correlations = {label: scaled[col].corr(scaled["NPS RATING"])
                    for label, col in metrics_zafran.items()}

    yearly = scaled.groupby("YEAR")[list(metrics_zafran.values())].mean().reset_index()

    fig = go.Figure()
    for label, col in metrics_zafran.items():
        is_nps = label == "NPS Rating"
        display_label = label if is_nps else label + " (x2 scaled)"
        fig.add_trace(go.Scatter(
            x=yearly["YEAR"],
            y=yearly[col],
            mode="lines+markers",
            name=display_label,
            # The NPS rating is the line the other four are being compared
            # against, so it is drawn solid and heavier than the sub-scores.
            line={"color": metric_colours_zafran[label],
                  "width": 3 if is_nps else 2,
                  "dash": "solid" if is_nps else "dot"},
            marker={"size": 7},
            hovertemplate=("<b>" + display_label + "</b> %{y:.2f}"
                           "<br>correlation with NPS rating r = "
                           + format(correlations[label], ".2f") + "<extra></extra>"),
        ))

    fig.update_layout(
        title={"text": "Which parts of the service track the NPS rating?",
               "x": 0.5, "xanchor": "center", "y": 0.96, "yanchor": "top"},
        # The y axis is left to scale itself. Every line sits between 7 and 8.5,
        # so pinning the axis to the full 0 to 10 range would flatten the trend
        # this chart exists to show.
        xaxis={"title": "Year", "tickmode": "linear", "dtick": 1,
               "rangeslider": {"visible": bool(show_buttons)}},
        yaxis={"title": "Score (0 to 10 scale, sub-scores x2)"},
        hovermode="x unified",
        legend={"title": "Metric, click to toggle"},
        plot_bgcolor="white",
        # An explicit width for the notebook version. Without one the chart is
        # drawn at whatever width the output area happens to report first, and
        # only snaps to the full width once an interaction forces a redraw. The
        # dashboard leaves it unset so the chart fills its panel.
        autosize=True,
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        margin={"t": 90 if show_buttons else 46, "b": 60},
    )

    return fig


fig_nps_trend_zafran(base_df_zafran, show_buttons=True).show()


### Insights

1. Technical Expertise (r ≈ 0.56) and Project Delivery (r ≈ 0.55) have the strongest correlations with NPS Rating, making them the service areas most closely associated with overall satisfaction.

2. Post-Sales Support (r ≈ 0.46) and Presales & Partnership (r ≈ 0.44) show weaker relationships with NPS Rating.

3. Presales & Partnership has the weakest correlation of the four sub-scores. Technical Expertise and Project Delivery are therefore stronger areas to investigate when examining what may be driving changes in NPS.

## DB 3 (Justin)

### Research Question

**How has the company's client growth between 2021 and 2025 translated into revenue growth, and is that growth broad-based across different client segments or increasingly dependent on a small number of high-value clients?**

#### Graph 1

### Business Question

**How did client acquisition grow between 2021 and 2025, and which client segments drove that growth?**

In [34]:
# ============================================================
# Cell: Graph 1 - Data Preparation
# ============================================================

# Client-level data for acquisition, and client-year revenue data for Graphs 2-3.
base_clients_justin = profiles.copy()
base_clients_justin["SIGNUP YEAR"] = pd.to_datetime(
    base_clients_justin["COMMENCEMENT DATE"]
).dt.year
base_clients_justin = base_clients_justin[
    base_clients_justin["SIGNUP YEAR"].between(2021, 2025)
].copy()
base_clients_justin["REGION"] = base_clients_justin["SG-BASED"].map(
    {"Y": "Singapore", "N": "Overseas"}
)

base_revenue_justin = RC.merge(
    profiles[["CLIENT ID", "TYPE", "SG-BASED", "SECTOR", "org_size"]],
    on="CLIENT ID",
    how="left"
)
assert len(base_revenue_justin) == len(RC), "the profiles join changed the row count"

g1_views_justin = {
    "Client Type": {
        "col": "TYPE",
        "order": ["Private", "Govt", "NPO"],
        "colours": {"Private": "#1F6FB4", "Govt": "#2CA7A4", "NPO": "#E06C39"},
    },
    "Region": {
        "col": "REGION",
        "order": ["Singapore", "Overseas"],
        "colours": {"Singapore": "#2CA7A4", "Overseas": "#B23A5F"},
    },
    "Organisation Size": {
        "col": "org_size",
        "order": ["Large", "Medium", "Small", "Unspecified"],
        "colours": {
            "Large": "#1F6FB4",
            "Medium": "#6BBF59",
            "Small": "#E8B73A",
            "Unspecified": "#A0A0A0",
        },
    },
}

In [35]:
# ============================================================
# Cell: Graph 1 - Plotly Express Chart
# ============================================================

import plotly.express as px


def acquisition_table_justin(data, view, years=None):
    """Count new clients by year and the selected segmentation view."""
    info = g1_views_justin[view]

    years = years or list(range(
        int(data["SIGNUP YEAR"].min()),
        int(data["SIGNUP YEAR"].max()) + 1
    ))

    counts = (
        data.groupby(["SIGNUP YEAR", info["col"]]).size()
        .rename("NEW CLIENTS").reset_index()
        .rename(columns={"SIGNUP YEAR": "YEAR", info["col"]: "SEGMENT"})
    )

    grid = pd.MultiIndex.from_product(
        [years, info["order"]],
        names=["YEAR", "SEGMENT"]
    ).to_frame(index=False)

    counts = grid.merge(counts, how="left").fillna({"NEW CLIENTS": 0})
    counts["NEW CLIENTS"] = counts["NEW CLIENTS"].astype(int)

    totals = counts.groupby("YEAR")["NEW CLIENTS"].transform("sum").replace(0, 1)
    counts["YEAR SHARE"] = counts["NEW CLIENTS"] / totals * 100

    return counts


def area_view_justin(data, view, years=None):
    """Build one Plotly Express stacked-area view."""
    info = g1_views_justin[view]
    counts = acquisition_table_justin(data, view, years)

    fig = px.area(
        counts,
        x="YEAR",
        y="NEW CLIENTS",
        color="SEGMENT",
        color_discrete_map=info["colours"],
        category_orders={"SEGMENT": info["order"]},
        custom_data=["YEAR SHARE"],
    )

    fig.update_traces(
        opacity=0.90,
        line_width=1.5,
        hovertemplate=(
            "<b>%{fullData.name}</b><br>"
            "Year: %{x}<br>"
            "New clients: %{y}<br>"
            "Share of year's acquisitions: %{customdata[0]:.1f}%"
            "<extra></extra>"
        ),
    )

    return fig


def fig_acquisition_justin(
    data,
    view="Client Type",
    show_buttons=False,
    years=None
):
    """Stacked area chart of new-client growth and the segments behind it."""
    if data.empty:
        return blank_figure("No new clients match these filters")

    views = list(g1_views_justin) if show_buttons else [view]

    fig = area_view_justin(data, views[0], years)
    groups = {views[0]: list(range(len(fig.data)))}

    for name in views[1:]:
        extra = area_view_justin(data, name, years)
        start = len(fig.data)

        for trace in extra.data:
            trace.visible = False

        fig.add_traces(extra.data)
        groups[name] = list(range(start, len(fig.data)))

    year_values = years or list(range(
        int(data["SIGNUP YEAR"].min()),
        int(data["SIGNUP YEAR"].max()) + 1
    ))

    totals = (
        data.groupby("SIGNUP YEAR").size()
        .reindex(year_values, fill_value=0)
        .rename("NEW CLIENTS")
        .reset_index()
    )

    total_trace = px.line(
        totals,
        x="SIGNUP YEAR",
        y="NEW CLIENTS",
        markers=True,
        text="NEW CLIENTS"
    ).data[0]

    total_trace.update(
        name="Total new clients",
        mode="lines+markers+text",
        textposition="top center",
        line={"color": "#08306B", "width": 4},
        marker={"size": 9, "color": "#08306B"},
        hovertemplate=(
            "Year: %{x}<br>"
            "Total new clients: %{y}"
            "<extra></extra>"
        ),
    )

    fig.add_trace(total_trace)
    total_index = len(fig.data) - 1

    fig.update_layout(
        title={
            "text": f"Client Acquisition Growth by {view}",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.96,
            "yanchor": "top",
        },
        xaxis={"title": "Year", "dtick": 1},
        yaxis={"title": "New Clients Acquired", "rangemode": "tozero"},
        legend={
            "title": view,
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.16,
        },
        hovermode="x unified",
        plot_bgcolor="white",
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        margin={
            "t": 140 if show_buttons else 46,
            "b": 100 if show_buttons else 70,
        },
    )

    if show_buttons:
        buttons = []

        for name in views:
            visible = [
                i in groups[name] or i == total_index
                for i in range(len(fig.data))
            ]

            buttons.append({
                "label": name,
                "method": "update",
                "args": [
                    {"visible": visible},
                    {
                        "title.text": f"Client Acquisition Growth by {name}",
                        "legend.title.text": name,
                    },
                ],
            })

        fig.update_layout(
            updatemenus=[{
                "type": "buttons",
                "direction": "right",
                "buttons": buttons,
                "x": 0.5,
                "xanchor": "center",
                "y": 1.10,
                "yanchor": "top",
                "showactive": True,
            }],
        )

    return fig


fig_acquisition_justin(
    base_clients_justin,
    show_buttons=True
).show()

### Key Insights

1. Client acquisition increased from 6 new clients in 2021 to a peak of 44 in 2024 before moderating to 26 in 2025, showing rapid expansion followed by a slowdown in the most recent year.

2. The Client Type view shows that Private clients drove most of this expansion, contributing 92 of the 124 new clients acquired from 2021 to 2025, or approximately 74% of total acquisition.

3. The Region and Organisation Size views show that growth was also concentrated towards Singapore-based clients and medium-to-large organisations. Around 80% of new clients were Singapore-based and 87% were medium or large organisations, showing that acquisition growth was strong but not evenly distributed across all client segments.

#### Graph 2

### Business Question

**Which client sectors generated the most revenue as the company grew between 2021 and 2025?**

In [36]:
# ============================================================
# Cell: Graph 2 - Data Preparation
# ============================================================

sector_order_justin = (
    base_revenue_justin.groupby("SECTOR")["REVENUE"]
    .sum()
    .sort_values()
    .index
    .tolist()
)

sector_palette_justin = [
    "#8DD3C7", "#FFFFB3", "#BEBADA", "#FB8072", "#80B1D3",
    "#FDB462", "#B3DE69", "#FCCDE5", "#D9D9D9", "#BC80BD"
]

sector_colours_justin = {
    sector: sector_palette_justin[i % len(sector_palette_justin)]
    for i, sector in enumerate(sorted(sector_order_justin))
}

In [37]:
# ============================================================
# Cell: Graph 2 - Plotly Graph Objects Chart
# ============================================================

import plotly.graph_objects as go


def sector_bar_justin(data, order=None):
    """One horizontal sector revenue ranking."""
    order = order or [
        s for s in sector_order_justin
        if s in data["SECTOR"].dropna().unique()
    ]

    sector = (
        data.groupby("SECTOR")["REVENUE"].sum()
        .reindex(order, fill_value=0)
        .rename("REVENUE")
        .reset_index()
    )

    total = sector["REVENUE"].sum()
    sector["SHARE"] = sector["REVENUE"] / total * 100 if total else 0

    return go.Bar(
        x=sector["REVENUE"],
        y=sector["SECTOR"],
        orientation="h",
        marker={
            "color": [sector_colours_justin[s] for s in sector["SECTOR"]],
            "line": {"color": "white", "width": 1},
        },
        text=[
            f"${v / 1_000_000:.1f}M" if v >= 1_000_000
            else f"${v / 1_000:.0f}k"
            for v in sector["REVENUE"]
        ],
        textposition="outside",
        customdata=sector[["SHARE"]].values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Revenue: $%{x:,.0f}<br>"
            "Share of selected revenue: %{customdata[0]:.1f}%"
            "<extra></extra>"
        ),
    )


def fig_sector_revenue_justin(data, show_buttons=False):
    """Sector revenue ranking; notebook version adds a year animation."""
    if data.empty:
        return blank_figure("No revenue matches these filters")

    years = sorted(data["YEAR"].unique())
    order = [
        s for s in sector_order_justin
        if s in data["SECTOR"].dropna().unique()
    ]

    first_year = years[0]
    first = data[data["YEAR"] == first_year] if show_buttons else data

    fig = go.Figure([
        sector_bar_justin(first, order)
    ])

    if show_buttons:
        fig.frames = [
            go.Frame(
                name=str(year),
                data=[
                    sector_bar_justin(
                        data[data["YEAR"] == year],
                        order
                    )
                ],
                layout=go.Layout(
                    title_text=f"Which sectors drive revenue? - {year}"
                ),
            )
            for year in years
        ]

    max_revenue = max(
        data.groupby(["YEAR", "SECTOR"])["REVENUE"].sum().max()
        if show_buttons
        else data.groupby("SECTOR")["REVENUE"].sum().max(),
        1
    )

    period = (
        str(first_year)
        if show_buttons or len(years) == 1
        else f"{years[0]}–{years[-1]}"
    )

    fig.update_layout(
        title={
            "text": f"Which sectors drive revenue? - {period}",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.96,
            "yanchor": "top",
        },
        xaxis={
            "title": "Revenue ($)",
            "range": [0, max_revenue * 1.18],
            "tickprefix": "$",
            "tickformat": "~s",
            "gridcolor": "rgba(0,0,0,0.08)",
        },
        yaxis={"title": "Client Sector"},
        plot_bgcolor="white",
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        margin={
            "t": 140 if show_buttons else 46,
            "b": 100,
            "l": 100,
            "r": 70,
        },
    )

    if show_buttons:
        steps = [{
            "label": str(year),
            "method": "animate",
            "args": [
                [str(year)],
                {
                    "mode": "immediate",
                    "frame": {"duration": 500, "redraw": True},
                    "transition": {"duration": 300},
                },
            ],
        } for year in years]

        fig.update_layout(
            sliders=[{
                "active": 0,
                "currentvalue": {"prefix": "Selected year: "},
                "pad": {"t": 35},
                "steps": steps,
            }],
            updatemenus=[{
                "type": "buttons",
                "direction": "right",
                "x": 0.5,
                "xanchor": "center",
                "y": 1.10,
                "yanchor": "top",
                "showactive": False,
                "buttons": [
                    {
                        "label": "▶ Play years",
                        "method": "animate",
                        "args": [
                            None,
                            {
                                "fromcurrent": True,
                                "frame": {"duration": 900, "redraw": True},
                                "transition": {"duration": 350},
                            },
                        ],
                    },
                    {
                        "label": "Ⅱ Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {
                                "mode": "immediate",
                                "frame": {"duration": 0, "redraw": False},
                                "transition": {"duration": 0},
                            },
                        ],
                    },
                ],
            }],
        )

    return fig


fig_sector_revenue_justin(
    base_revenue_justin,
    show_buttons=True
).show()

### Key Insights

1. Total company revenue increased from approximately $7.6 million in 2021 to $58.8 million in 2025, representing almost 7.8 times the 2021 revenue level.

2. The year slider shows that revenue leadership changed over time. Healthcare led in 2021, 2023 and 2024, Manufacturing led in 2022, while Info Tech became the largest sector in 2025 with approximately $14.9 million in revenue.

3. In 2025, Info Tech, Healthcare and Manufacturing together contributed approximately 61% of annual revenue. However, meaningful revenue was also generated across several other sectors, so the next analysis tests whether this apparent sector diversification is also reflected at individual-client level.

#### Graph 3

### Business Question

Is the company's revenue concentrated among a small number of high-value clients?

In [38]:
# ============================================================
# Cell: Graph 3 - Data Preparation
# ============================================================

PARETO_TEAL_justin = "#4FB3A5"
PARETO_GREY_justin = "#D9D9D9"
CUMULATIVE_ORANGE_justin = "#E6550D"
REFERENCE_RED_justin = "#D62728"


def pareto_data_justin(data):
    """Rank clients by revenue and calculate cumulative revenue share."""
    clients = (
        data.groupby("CLIENT ID", as_index=False)["REVENUE"]
        .sum()
        .sort_values("REVENUE", ascending=False)
        .reset_index(drop=True)
    )

    clients["CUMULATIVE %"] = (
        clients["REVENUE"].cumsum()
        / clients["REVENUE"].sum()
        * 100
    )

    clients_to_80 = clients["CUMULATIVE %"].ge(80).idxmax() + 1

    return clients, clients_to_80

In [39]:
# ============================================================
# Cell: Graph 3 - Plotly Graph Objects Chart
# ============================================================

import plotly.graph_objects as go


def pareto_pair_justin(data, visible=True):
    """Revenue bars and cumulative-share line for one selected period."""
    clients, clients_to_80 = pareto_data_justin(data)
    ranks = list(range(1, len(clients) + 1))

    colours = [
        PARETO_TEAL_justin if rank <= clients_to_80 else PARETO_GREY_justin
        for rank in ranks
    ]

    bars = go.Bar(
        x=ranks,
        y=clients["REVENUE"],
        name="Client revenue",
        visible=visible,
        marker_color=colours,
        customdata=clients["CLIENT ID"],
        hovertemplate=(
            "Rank %{x}<br>"
            "Client %{customdata}<br>"
            "Revenue: $%{y:,.0f}"
            "<extra></extra>"
        ),
    )

    line = go.Scatter(
        x=ranks,
        y=clients["CUMULATIVE %"],
        name="Cumulative revenue",
        mode="lines",
        yaxis="y2",
        visible=visible,
        line={
            "color": CUMULATIVE_ORANGE_justin,
            "width": 3,
        },
        hovertemplate=(
            "Rank %{x}<br>"
            "Cumulative revenue: %{y:.1f}%"
            "<extra></extra>"
        ),
    )

    return bars, line


def pareto_overlay_justin(data):
    """80% reference lines and annotation for one selected period."""
    clients, clients_to_80 = pareto_data_justin(data)
    share = clients_to_80 / len(clients) * 100

    shapes = [
        {
            "type": "line",
            "xref": "paper",
            "x0": 0,
            "x1": 1,
            "yref": "y2",
            "y0": 80,
            "y1": 80,
            "line": {
                "color": REFERENCE_RED_justin,
                "dash": "dash",
                "width": 2,
            },
        },
        {
            "type": "line",
            "xref": "x",
            "x0": clients_to_80,
            "x1": clients_to_80,
            "yref": "paper",
            "y0": 0,
            "y1": 1,
            "line": {
                "color": REFERENCE_RED_justin,
                "dash": "dash",
                "width": 2,
            },
        },
    ]

    annotations = [{
        "x": clients_to_80,
        "y": 84,
        "yref": "y2",
        "text": (
            f"80% reached after {clients_to_80} of {len(clients)} clients"
            f"<br>({share:.0f}% of the client base)"
        ),
        "showarrow": True,
        "arrowhead": 2,
        "ax": 80,
        "ay": -45,
        "arrowcolor": REFERENCE_RED_justin,
        "bgcolor": "#FFF3E8",
        "bordercolor": CUMULATIVE_ORANGE_justin,
        "borderwidth": 1,
    }]

    return shapes, annotations


def fig_pareto_justin(data, show_buttons=False):
    """Pareto chart; notebook version can switch between years."""
    if data.empty:
        return blank_figure("No revenue matches these filters")

    years = sorted(data["YEAR"].unique())
    periods = ["All Years"] + years if show_buttons else ["Selected Period"]

    fig = go.Figure()
    period_data = {}

    for i, period in enumerate(periods):
        subset = (
            data
            if period in ["All Years", "Selected Period"]
            else data[data["YEAR"] == period]
        )

        period_data[period] = subset
        fig.add_traces(
            pareto_pair_justin(
                subset,
                visible=(i == 0)
            )
        )

    shapes, annotations = pareto_overlay_justin(
        period_data[periods[0]]
    )

    clients, _ = pareto_data_justin(
        period_data[periods[0]]
    )

    title_period = (
        "All Years"
        if show_buttons
        else str(years[0])
        if len(years) == 1
        else f"{years[0]}–{years[-1]}"
    )

    fig.update_layout(
        title={
            "text": f"How concentrated is client revenue? - {title_period}",
            "x": 0.5,
            "xanchor": "center",
            "y": 0.96,
            "yanchor": "top",
        },
        xaxis={
            "title": "Clients ranked by revenue",
            "range": [0, len(clients) + 1],
            "showgrid": False,
        },
        yaxis={
            "title": "Revenue ($)",
            "tickprefix": "$",
            "tickformat": "~s",
            "gridcolor": "rgba(0,0,0,0.08)",
        },
        yaxis2={
            "title": "Cumulative revenue (%)",
            "overlaying": "y",
            "side": "right",
            "range": [0, 100],
            "ticksuffix": "%",
        },
        legend={
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.15,
        },
        hovermode="x unified",
        plot_bgcolor="white",
        height=700 if show_buttons else DASH_CHART_HEIGHT,
        margin={
            "t": 140 if show_buttons else 46,
            "b": 85,
            "l": 70,
            "r": 80,
        },
        shapes=shapes,
        annotations=annotations if show_buttons else [],
    )

    if show_buttons:
        buttons = []

        for i, period in enumerate(periods):
            subset = period_data[period]

            period_shapes, period_annotations = pareto_overlay_justin(subset)
            period_clients, _ = pareto_data_justin(subset)

            buttons.append({
                "label": str(period),
                "method": "update",
                "args": [
                    {
                        "visible": [
                            j // 2 == i
                            for j in range(len(periods) * 2)
                        ]
                    },
                    {
                        "title.text":
                            f"How concentrated is client revenue? - {period}",
                        "xaxis.range": [0, len(period_clients) + 1],
                        "shapes": period_shapes,
                        "annotations": period_annotations,
                    },
                ],
            })

        fig.update_layout(
            updatemenus=[{
                "type": "buttons",
                "direction": "right",
                "buttons": buttons,
                "x": 0.5,
                "xanchor": "center",
                "y": 1.10,
                "yanchor": "top",
                "showactive": True,
            }],
        )

    return fig


fig_pareto_justin(
    base_revenue_justin,
    show_buttons=True
).show()

### Key Insights

1. Across 2021–2025, 66 of 140 clients, or approximately 47% of the client base, were required to generate 80% of total revenue. This does not resemble an extreme 80/20 concentration where only a small group carries the business.

2. The year buttons show that revenue became less dependent on the largest accounts as the company expanded. The top 10 clients contributed approximately 69% of revenue in 2021 but only about 20% in 2025.

3. The single largest client contributed only about 2.8% of total five-year revenue. This suggests that retention efforts should protect a broad group of valuable clients rather than relying mainly on a small number of exceptionally large accounts.

# Group Storyboard (Plotly Dash)

One Dash app holding all three members' dashboards, with radio buttons under the
header to switch between them.

The app is built in the usual five parts: instantiate the app, write the layout,
add the callbacks, write the function each callback calls, then run it.

**How the three pages fit together.** Each member has one `html.Div` in the
layout with an id ending in `_page`, and their own callbacks further down. The
page switch just shows one and hides the other two. Two naming rules keep three
people's code from clashing in one notebook:

1. Every component id starts with the member's name, e.g. `elliot_graph1`.
2. Every variable a callback reads ends with the member's name, e.g.
   `metrics_elliot`. A callback reads variables when the user clicks, not when
   the cell runs, so without this the last cell in the notebook would win.

Zafran's and Justin's pages are placeholders for now.


In [40]:
# ---- Imports and shared styles for the dashboard ----
# Note: app.run_server() was removed in Dash 4, so the last cell uses app.run().
import dash
from dash import dcc, html, Input, Output

# Height of each chart inside the dashboard. The whole page is built to fit on
# one screen at 1920x1080 with the browser full screen, so change this one
# number if it does not quite fit on the machine you are presenting from.
DASH_CHART_HEIGHT = 335

# Plain style dictionaries instead of a stylesheet, passed to style= below.
# boxSizing tells the browser to count padding and border inside the width we
# ask for. Without it a card set to 18% is really 18% plus its padding, and five
# of them no longer fit on one row.
HEADER_STYLE = {"backgroundColor": "#0f3d5c", "color": "white",
                "padding": "10px 24px", "display": "flex",
                "justifyContent": "space-between", "alignItems": "center"}
PANEL_STYLE = {"backgroundColor": "white", "border": "1px solid #dddddd",
               "borderRadius": "8px", "padding": "10px 12px",
               "marginBottom": "10px", "boxSizing": "border-box"}
FIELD_STYLE = {"display": "inline-block", "width": "20%",
               "verticalAlign": "top", "paddingRight": "12px",
               "boxSizing": "border-box"}
KPI_STYLE = {"display": "inline-block", "width": "20%", "backgroundColor": "white",
             "border": "1px solid #dddddd", "borderRadius": "8px",
             "padding": "8px 10px", "verticalAlign": "top",
             "boxSizing": "border-box"}
LABEL_STYLE = {"fontSize": "11px", "color": "#666666", "fontWeight": "bold"}


def kpi_card(title, value, note):
    """One KPI card. Shared by every member's page, so the row of cards looks
    the same whichever dashboard is showing."""
    return html.Div([
        html.Div(title, style=LABEL_STYLE),
        html.Div(value, style={"fontSize": "20px", "fontWeight": "bold"}),
        html.Div(note, style={"fontSize": "11px", "color": "#666666"}),
    ], style=KPI_STYLE)


In [41]:
# ---- Elliot: filtering and the KPI number formatting ----
# The card itself is shared, and lives with the styles above.

years_elliot = sorted(base_df_elliot["YEAR"].unique())
types_elliot = ["Private", "Govt", "NPO"]
sizes_elliot = ["Small", "Medium", "Large", "Unspecified"]
sectors_elliot = sorted(base_df_elliot["SECTOR"].unique())


def apply_filters_elliot(years, types, sectors, sizes, region):
    """Cut the data down to what the navigation bar is asking for.

    All three charts and the KPI cards call this, so they can never end up
    showing different selections.
    """
    data = base_df_elliot[base_df_elliot["YEAR"].between(years[0], years[1])]

    # An empty list means the user unticked everything, so show nothing at all
    # rather than quietly treating it as "show all".
    data = data[data["TYPE"].isin(types)]

    if sectors:
        data = data[data["SECTOR"].isin(sectors)]
    if sizes:
        data = data[data["org_size"].isin(sizes)]
    if region == "Singapore":
        data = data[data["SG-BASED"] == "Y"]
    elif region == "Overseas":
        data = data[data["SG-BASED"] == "N"]
    return data


def money_elliot(value):
    """Shorten a dollar amount so it fits on one line of a KPI card."""
    if value >= 1000000:
        return "$" + format(value / 1000000, ",.1f") + "M"
    if value >= 1000:
        return "$" + format(value / 1000, ",.0f") + "k"
    return "$" + format(value, ",.0f")


In [42]:
# ---- Zafran: filtering and the KPI figures ----

years_zafran = sorted(base_df_zafran["YEAR"].unique())
types_zafran = ["Private", "Govt", "NPO"]
sizes_zafran = ["Small", "Medium", "Large", "Unspecified"]


def apply_filters_zafran(years, types, sectors, sizes, region):
    """Cut the survey data down to what the filter bar is asking for.

    All three charts and the KPI cards call this, so they can never end up
    showing different selections.
    """
    data = base_df_zafran[base_df_zafran["YEAR"].between(years[0], years[1])]

    # An empty list means the user unticked everything, so show nothing at all
    # rather than quietly treating it as "show all".
    data = data[data["TYPE"].isin(types)]

    if sectors:
        data = data[data["SECTOR"].isin(sectors)]
    if sizes:
        data = data[data["org_size"].isin(sizes)]
    if region == "Singapore":
        data = data[data["SG-BASED"] == "Y"]
    elif region == "Overseas":
        data = data[data["SG-BASED"] == "N"]
    return data


def nps_score_zafran(data):
    """The Net Promoter Score itself: the share of Promoters less the share of
    Detractors, which is how the industry defines the metric. Passives count
    towards the base but not towards the score."""
    if data.empty:
        return 0
    promoters = (data["nps_category"] == "Promoter").mean() * 100
    detractors = (data["nps_category"] == "Detractor").mean() * 100
    return promoters - detractors


def share_zafran(data, category):
    """What percentage of these responses fall in one NPS category."""
    if data.empty:
        return 0
    return (data["nps_category"] == category).mean() * 100


In [43]:
# ---- Justin: filtering and the KPI number formatting ----

years_dashboard_justin = sorted(base_revenue_justin["YEAR"].unique())
types_dashboard_justin = ["Private", "Govt", "NPO"]
sizes_dashboard_justin = ["Small", "Medium", "Large", "Unspecified"]
sectors_dashboard_justin = sorted(
    base_revenue_justin["SECTOR"].dropna().unique()
)


def apply_filters_justin(years, types, sectors, sizes, region):
    """Apply the same filter bar selections to revenue and acquisition data."""
    revenue = base_revenue_justin[
        base_revenue_justin["YEAR"].between(years[0], years[1])
    ].copy()

    clients = base_clients_justin[
        base_clients_justin["SIGNUP YEAR"].between(years[0], years[1])
    ].copy()

    def filter_segments(data):
        data = data[data["TYPE"].isin(types)]

        if sectors:
            data = data[data["SECTOR"].isin(sectors)]

        if sizes:
            data = data[data["org_size"].isin(sizes)]

        if region == "Singapore":
            data = data[data["SG-BASED"] == "Y"]
        elif region == "Overseas":
            data = data[data["SG-BASED"] == "N"]

        return data

    return filter_segments(revenue), filter_segments(clients)


def money_justin(value):
    """Shorten a dollar amount so it fits on one KPI card."""
    if value >= 1_000_000:
        return "$" + format(value / 1_000_000, ",.1f") + "M"

    if value >= 1_000:
        return "$" + format(value / 1_000, ",.0f") + "k"

    return "$" + format(value, ",.0f")

In [44]:
# ---- PART 1: instantiate the app ----
app = dash.Dash(__name__)
app.title = "IT Systems Integrator: Client Analytics"   # browser tab name

# ---- PART 2: the layout ----
# The whole page is built to fit one screen with no scrolling, so that a single
# screenshot shows the entire dashboard.
app.layout = html.Div([

    # Header: title on the left, and the navigation for whose dashboard to show
    # on the right. Both sit on one line to save vertical space.
    html.Div([
        html.Div([
            html.H3("IT Systems Integrator: Client Analytics",
                    style={"margin": "0px"}),
            html.Div("Is the company's rapid client growth (2021-2025) "
                     "profitable and sustainable?",
                     style={"fontSize": "12px"}),
        ]),
        dcc.RadioItems(
            id="member_switch",
            options=[{"label": " Elliot", "value": "elliot"},
                     {"label": " Zafran", "value": "zafran"},
                     {"label": " Justin", "value": "justin"}],
            value="elliot",
            inline=True,
            inputStyle={"marginRight": "6px", "accentColor": "#4fb3a5"},
            labelStyle={"marginRight": "18px", "fontSize": "14px", "color": "white"},
        ),
    ], style=HEADER_STYLE),

    # ---- Main content area: Elliot's page ----
    html.Div([

        # Filter bar. Five controls across one row.
        html.Div([
            html.Div([
                html.Label(["Year range"], style=LABEL_STYLE),
                dcc.RangeSlider(
                    id="elliot_year_range",
                    min=years_elliot[0], max=years_elliot[-1], step=1,
                    value=[years_elliot[0], years_elliot[-1]],
                    marks={int(year): str(year) for year in years_elliot},
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Client type"], style=LABEL_STYLE),
                dcc.Checklist(
                    id="elliot_type",
                    options=[{"label": " " + t, "value": t} for t in types_elliot],
                    value=types_elliot,
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "9px", "fontSize": "13px"},
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Sector"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="elliot_sector",
                    options=[{"label": s, "value": s} for s in sectors_elliot],
                    multi=True,
                    placeholder="All sectors",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Organisation size"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="elliot_orgsize",
                    options=[{"label": s, "value": s} for s in sizes_elliot],
                    multi=True,
                    placeholder="All sizes",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Region"], style=LABEL_STYLE),
                dcc.RadioItems(
                    id="elliot_region",
                    options=[{"label": " All", "value": "All"},
                             {"label": " Singapore", "value": "Singapore"},
                             {"label": " Overseas", "value": "Overseas"}],
                    value="All",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "8px", "fontSize": "13px"},
                ),
            ], style=FIELD_STYLE),
        ], style=PANEL_STYLE),

        # KPI cards, filled in by a callback
        html.Div(id="elliot_kpis", style={"marginBottom": "10px"}),

        # Charts 1 and 2, side by side. Graph 1 is given a little more width
        # because it has a legend with eight entries. minWidth of 0 is needed or
        # a chart inside a flex box refuses to shrink and pushes the row wider
        # than the screen.
        html.Div([
            html.Div([
                dcc.RadioItems(
                    id="elliot_g1_view",
                    options=[{"label": " " + v, "value": v} for v in g1_views_elliot],
                    value="Gross profit per client",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "12px", "fontSize": "13px"},
                ),
                dcc.Graph(id="elliot_graph1", config={"displaylogo": False}),
            ], style={**PANEL_STYLE, "flex": "1.15", "minWidth": "0"}),

            html.Div([
                dcc.RadioItems(
                    id="elliot_g2_mode",
                    options=[{"label": " " + m, "value": m} for m in g2_modes_elliot],
                    value="Total $",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "12px", "fontSize": "13px"},
                ),
                dcc.Graph(id="elliot_graph2", config={"displaylogo": False}),
            ], style={**PANEL_STYLE, "flex": "1", "minWidth": "0"}),
        ], style={"display": "flex", "gap": "10px"}),

        # Chart 3 gets the full width because it has ten sectors along the x axis
        html.Div([
            dcc.RadioItems(
                id="elliot_g3_metric",
                options=[{"label": " " + m, "value": m} for m in metrics_elliot],
                value="Gross Margin %",
                inline=True,
                inputStyle={"marginRight": "4px"},
                labelStyle={"marginRight": "12px", "fontSize": "13px"},
            ),
            dcc.Graph(id="elliot_graph3", config={"displaylogo": False}),
        ], style=PANEL_STYLE),

    ], id="elliot_page"),

    # ---- Main content area: Zafran's page ----
    html.Div([

        # Filter bar. The same five controls as Elliot's page, so the storyboard
        # reads as one product rather than three separate apps. There is no NPS
        # category filter on purpose: Promoter, Passive and Detractor are what
        # the first two charts are made of, so filtering on it would leave a
        # single band and nothing to compare.
        html.Div([
            html.Div([
                html.Label(["Year range"], style=LABEL_STYLE),
                dcc.RangeSlider(
                    id="zafran_year_range",
                    min=years_zafran[0], max=years_zafran[-1], step=1,
                    value=[years_zafran[0], years_zafran[-1]],
                    marks={int(year): str(year) for year in years_zafran},
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Client type"], style=LABEL_STYLE),
                dcc.Checklist(
                    id="zafran_type",
                    options=[{"label": " " + t, "value": t} for t in types_zafran],
                    value=types_zafran,
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "9px", "fontSize": "13px"},
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Sector"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="zafran_sector",
                    options=[{"label": s, "value": s} for s in sectors_all_zafran],
                    multi=True,
                    placeholder="All sectors",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Organisation size"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="zafran_orgsize",
                    options=[{"label": s, "value": s} for s in sizes_zafran],
                    multi=True,
                    placeholder="All sizes",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Region"], style=LABEL_STYLE),
                dcc.RadioItems(
                    id="zafran_region",
                    options=[{"label": " All", "value": "All"},
                             {"label": " Singapore", "value": "Singapore"},
                             {"label": " Overseas", "value": "Overseas"}],
                    value="All",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "8px", "fontSize": "13px"},
                ),
            ], style=FIELD_STYLE),
        ], style=PANEL_STYLE),

        # KPI cards, filled in by a callback
        html.Div(id="zafran_kpis", style={"marginBottom": "10px"}),

        # Charts 1 and 2 side by side. The sunburst has no view selector of its
        # own, because the year slider it carries in the notebook is what the
        # year range on the filter bar now does.
        html.Div([
            html.Div([
                dcc.Graph(id="zafran_graph1", config={"displaylogo": False}),
            ], style={**PANEL_STYLE, "flex": "1", "minWidth": "0"}),

            html.Div([
                dcc.RadioItems(
                    id="zafran_g2_mode",
                    options=[{"label": " " + m, "value": m} for m in g2_modes_zafran],
                    value="Raw counts",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={"marginRight": "12px", "fontSize": "13px"},
                ),
                dcc.Graph(id="zafran_graph2", config={"displaylogo": False}),
            ], style={**PANEL_STYLE, "flex": "1.15", "minWidth": "0"}),
        ], style={"display": "flex", "gap": "10px"}),

        # Chart 3 gets the full width because it draws five lines with a five
        # entry legend and reads them all in one hover.
        html.Div([
            dcc.Graph(id="zafran_graph3", config={"displaylogo": False}),
        ], style=PANEL_STYLE),

    ], id="zafran_page", style={"display": "none"}),

    # ---- Main content area: Justin's page ----
    html.Div([

        # Filter bar - same five controls as the other dashboard pages.
        html.Div([
            html.Div([
                html.Label(["Year range"], style=LABEL_STYLE),
                dcc.RangeSlider(
                    id="justin_year_range",
                    min=years_dashboard_justin[0],
                    max=years_dashboard_justin[-1],
                    step=1,
                    value=[
                        years_dashboard_justin[0],
                        years_dashboard_justin[-1]
                    ],
                    marks={
                        int(year): str(year)
                        for year in years_dashboard_justin
                    },
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Client type"], style=LABEL_STYLE),
                dcc.Checklist(
                    id="justin_type",
                    options=[
                        {"label": " " + t, "value": t}
                        for t in types_dashboard_justin
                    ],
                    value=types_dashboard_justin,
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={
                        "marginRight": "9px",
                        "fontSize": "13px"
                    },
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Sector"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="justin_sector",
                    options=[
                        {"label": s, "value": s}
                        for s in sectors_dashboard_justin
                    ],
                    multi=True,
                    placeholder="All sectors",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Organisation size"], style=LABEL_STYLE),
                dcc.Dropdown(
                    id="justin_orgsize",
                    options=[
                        {"label": s, "value": s}
                        for s in sizes_dashboard_justin
                    ],
                    multi=True,
                    placeholder="All sizes",
                ),
            ], style=FIELD_STYLE),

            html.Div([
                html.Label(["Region"], style=LABEL_STYLE),
                dcc.RadioItems(
                    id="justin_region",
                    options=[
                        {"label": " All", "value": "All"},
                        {"label": " Singapore", "value": "Singapore"},
                        {"label": " Overseas", "value": "Overseas"},
                    ],
                    value="All",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={
                        "marginRight": "8px",
                        "fontSize": "13px"
                    },
                ),
            ], style=FIELD_STYLE),

        ], style=PANEL_STYLE),

        # KPI cards, filled in by a callback.
        html.Div(
            id="justin_kpis",
            style={"marginBottom": "10px"}
        ),

        # Charts 1 and 2 side by side.
        html.Div([
            html.Div([
                dcc.RadioItems(
                    id="justin_g1_view",
                    options=[
                        {"label": " " + view, "value": view}
                        for view in g1_views_justin
                    ],
                    value="Client Type",
                    inline=True,
                    inputStyle={"marginRight": "4px"},
                    labelStyle={
                        "marginRight": "12px",
                        "fontSize": "13px"
                    },
                ),
                dcc.Graph(
                    id="justin_graph1",
                    config={"displaylogo": False}
                ),
            ], style={
                **PANEL_STYLE,
                "flex": "1",
                "minWidth": "0"
            }),

            html.Div([
                dcc.Graph(
                    id="justin_graph2",
                    config={"displaylogo": False}
                ),
            ], style={
                **PANEL_STYLE,
                "flex": "1",
                "minWidth": "0"
            }),

        ], style={
            "display": "flex",
            "gap": "10px"
        }),

        # Pareto chart gets the full width.
        html.Div([
            dcc.Graph(
                id="justin_graph3",
                config={"displaylogo": False}
            ),
        ], style=PANEL_STYLE),

    ], id="justin_page", style={"display": "none"}),

], style={"backgroundColor": "#f4f6f7", "paddingBottom": "10px"})


In [45]:
# ---- PART 3 and 4: the callbacks and the functions they call ----

# Show the chosen member's page and hide the other two.
@app.callback(
    Output("elliot_page", "style"),
    Output("zafran_page", "style"),
    Output("justin_page", "style"),
    Input("member_switch", "value"),
)
def show_member_page(member):
    show = {"display": "block", "padding": "10px 14px"}
    hide = {"display": "none"}
    return (show if member == "elliot" else hide,
            show if member == "zafran" else hide,
            show if member == "justin" else hide)


# The KPI cards above Elliot's charts.
@app.callback(
    Output("elliot_kpis", "children"),
    Input("elliot_year_range", "value"),
    Input("elliot_type", "value"),
    Input("elliot_sector", "value"),
    Input("elliot_orgsize", "value"),
    Input("elliot_region", "value"),
)
def update_kpis_elliot(years, types, sectors, sizes, region):
    data = apply_filters_elliot(years, types, sectors, sizes, region)
    if data.empty:
        return kpi_card("No data", "-", "nothing matches these filters")

    whole = base_df_elliot
    revenue = data["REVENUE"].sum()
    profit = data["gross_profit"].sum()
    clients = data["CLIENT ID"].nunique()
    margin = profit / revenue * 100
    company_margin = whole["gross_profit"].sum() / whole["REVENUE"].sum() * 100

    # Profit per client is worked out for a single year on purpose. Dividing
    # several years of profit by a client count would mostly measure how long
    # clients have been with us, and would not match Graph 1.
    year = data["YEAR"].max()
    this_year = data[data["YEAR"] == year]
    company_year = whole[whole["YEAR"] == year]
    per_client = this_year["gross_profit"].sum() / this_year["CLIENT ID"].nunique()
    company_per_client = (company_year["gross_profit"].sum()
                          / company_year["CLIENT ID"].nunique())

    return [
        kpi_card("Revenue", money_elliot(revenue),
                        format(revenue / whole["REVENUE"].sum() * 100, ".0f")
                        + "% of company total"),
        kpi_card("Gross profit", money_elliot(profit),
                        format(profit / whole["gross_profit"].sum() * 100, ".0f")
                        + "% of company total"),
        kpi_card("Gross margin", format(margin, ".1f") + "%",
                        "company " + format(company_margin, ".1f") + "%"),
        kpi_card("Clients", format(clients, ","),
                        "of " + format(whole["CLIENT ID"].nunique(), ",")),
        kpi_card("Profit / client, " + str(year), money_elliot(per_client),
                        "company " + money_elliot(company_per_client)),
    ]


# Chart 1: profit per client by intake year.
@app.callback(
    Output("elliot_graph1", "figure"),
    Input("elliot_year_range", "value"),
    Input("elliot_type", "value"),
    Input("elliot_sector", "value"),
    Input("elliot_orgsize", "value"),
    Input("elliot_region", "value"),
    Input("elliot_g1_view", "value"),
)
def update_graph1_elliot(years, types, sectors, sizes, region, view):
    data = apply_filters_elliot(years, types, sectors, sizes, region)
    return fig_cohort_elliot(data, view=view)


# Chart 2: cost make-up and gross margin.
@app.callback(
    Output("elliot_graph2", "figure"),
    Input("elliot_year_range", "value"),
    Input("elliot_type", "value"),
    Input("elliot_sector", "value"),
    Input("elliot_orgsize", "value"),
    Input("elliot_region", "value"),
    Input("elliot_g2_mode", "value"),
)
def update_graph2_elliot(years, types, sectors, sizes, region, mode):
    data = apply_filters_elliot(years, types, sectors, sizes, region)
    return fig_cogs_elliot(data, mode=mode)


# Chart 3: distribution by sector.
@app.callback(
    Output("elliot_graph3", "figure"),
    Input("elliot_year_range", "value"),
    Input("elliot_type", "value"),
    Input("elliot_sector", "value"),
    Input("elliot_orgsize", "value"),
    Input("elliot_region", "value"),
    Input("elliot_g3_metric", "value"),
)
def update_graph3_elliot(years, types, sectors, sizes, region, metric):
    data = apply_filters_elliot(years, types, sectors, sizes, region)
    return fig_sector_elliot(data, metric_name=metric)


# The KPI cards above Zafran's charts.
@app.callback(
    Output("zafran_kpis", "children"),
    Input("zafran_year_range", "value"),
    Input("zafran_type", "value"),
    Input("zafran_sector", "value"),
    Input("zafran_orgsize", "value"),
    Input("zafran_region", "value"),
)
def update_kpis_zafran(years, types, sectors, sizes, region):
    data = apply_filters_zafran(years, types, sectors, sizes, region)
    if data.empty:
        return kpi_card("No data", "-", "nothing matches these filters")

    # The four satisfaction figures are read off the latest year in the range,
    # the same way Elliot's profit per client is. Averaging five years together
    # would hide the whole point of this page, which is that the scores move.
    year = data["YEAR"].max()
    latest = data[data["YEAR"] == year]
    company = base_df_zafran[base_df_zafran["YEAR"] == year]

    return [
        kpi_card("NPS score, " + str(year), format(nps_score_zafran(latest), ".0f"),
                 "company " + format(nps_score_zafran(company), ".0f")),
        kpi_card("Avg NPS rating, " + str(year),
                 format(latest["NPS RATING"].mean(), ".2f"),
                 "company " + format(company["NPS RATING"].mean(), ".2f")),
        kpi_card("Promoters, " + str(year),
                 format(share_zafran(latest, "Promoter"), ".0f") + "%",
                 "company " + format(share_zafran(company, "Promoter"), ".0f") + "%"),
        kpi_card("Detractors, " + str(year),
                 format(share_zafran(latest, "Detractor"), ".0f") + "%",
                 "company " + format(share_zafran(company, "Detractor"), ".0f") + "%"),
        kpi_card("Responses", format(len(data), ","),
                 "from " + format(data["CLIENT ID"].nunique(), ",") + " clients"),
    ]


# Chart 1: the Promoter, Passive and Detractor mix of each sector.
@app.callback(
    Output("zafran_graph1", "figure"),
    Input("zafran_year_range", "value"),
    Input("zafran_type", "value"),
    Input("zafran_sector", "value"),
    Input("zafran_orgsize", "value"),
    Input("zafran_region", "value"),
)
def update_graph1_zafran(years, types, sectors, sizes, region):
    data = apply_filters_zafran(years, types, sectors, sizes, region)
    return fig_nps_mix_zafran(data)


# Chart 2: the same mix year by year, as counts or as a share.
@app.callback(
    Output("zafran_graph2", "figure"),
    Input("zafran_year_range", "value"),
    Input("zafran_type", "value"),
    Input("zafran_sector", "value"),
    Input("zafran_orgsize", "value"),
    Input("zafran_region", "value"),
    Input("zafran_g2_mode", "value"),
)
def update_graph2_zafran(years, types, sectors, sizes, region, mode):
    data = apply_filters_zafran(years, types, sectors, sizes, region)
    return fig_nps_stack_zafran(data, mode=mode)


# Chart 3: the NPS rating against the four sub-scores behind it.
@app.callback(
    Output("zafran_graph3", "figure"),
    Input("zafran_year_range", "value"),
    Input("zafran_type", "value"),
    Input("zafran_sector", "value"),
    Input("zafran_orgsize", "value"),
    Input("zafran_region", "value"),
)
def update_graph3_zafran(years, types, sectors, sizes, region):
    data = apply_filters_zafran(years, types, sectors, sizes, region)
    return fig_nps_trend_zafran(data)


# The KPI cards above Justin's charts.
@app.callback(
    Output("justin_kpis", "children"),
    Input("justin_year_range", "value"),
    Input("justin_type", "value"),
    Input("justin_sector", "value"),
    Input("justin_orgsize", "value"),
    Input("justin_region", "value"),
)
def update_kpis_justin(years, types, sectors, sizes, region):
    revenue, clients = apply_filters_justin(
        years, types, sectors, sizes, region
    )

    if revenue.empty and clients.empty:
        return kpi_card(
            "No data",
            "-",
            "nothing matches these filters"
        )

    total_revenue = revenue["REVENUE"].sum()
    new_clients = clients["CLIENT ID"].nunique()

    if revenue.empty:
        top_sector, top_value = "-", 0
        clients_to_80, revenue_clients, largest_share = 0, 0, 0

    else:
        sector_totals = (
            revenue.groupby("SECTOR")["REVENUE"]
            .sum()
            .sort_values(ascending=False)
        )

        top_sector = sector_totals.index[0]
        top_value = sector_totals.iloc[0]

        client_totals = (
            revenue.groupby("CLIENT ID")["REVENUE"]
            .sum()
            .sort_values(ascending=False)
        )

        cumulative = (
            client_totals.cumsum()
            / client_totals.sum()
            * 100
        )

        clients_to_80 = (
            cumulative.ge(80).to_numpy().argmax() + 1
        )

        revenue_clients = len(client_totals)

        largest_share = (
            client_totals.iloc[0]
            / client_totals.sum()
            * 100
        )

    return [
        kpi_card(
            "New clients",
            format(new_clients, ","),
            f"{years[0]}-{years[1]} signings"
        ),

        kpi_card(
            "Revenue",
            money_justin(total_revenue),
            "selected period total"
        ),

        kpi_card(
            "Top sector",
            top_sector,
            money_justin(top_value) + " selected revenue"
        ),

        kpi_card(
            "Clients to 80% revenue",
            f"{clients_to_80} / {revenue_clients}"
            if revenue_clients else "-",
            f"{clients_to_80 / revenue_clients * 100:.0f}% of clients"
            if revenue_clients else "no revenue data"
        ),

        kpi_card(
            "Largest client share",
            f"{largest_share:.1f}%",
            "of selected revenue"
        ),
    ]


# Chart 1: new-client growth and its segment mix.
@app.callback(
    Output("justin_graph1", "figure"),
    Input("justin_year_range", "value"),
    Input("justin_type", "value"),
    Input("justin_sector", "value"),
    Input("justin_orgsize", "value"),
    Input("justin_region", "value"),
    Input("justin_g1_view", "value"),
)
def update_graph1_justin(
    years,
    types,
    sectors,
    sizes,
    region,
    view
):
    _, clients = apply_filters_justin(
        years, types, sectors, sizes, region
    )

    return fig_acquisition_justin(
        clients,
        view=view,
        years=list(range(years[0], years[1] + 1))
    )


# Chart 2: sector revenue.
@app.callback(
    Output("justin_graph2", "figure"),
    Input("justin_year_range", "value"),
    Input("justin_type", "value"),
    Input("justin_sector", "value"),
    Input("justin_orgsize", "value"),
    Input("justin_region", "value"),
)
def update_graph2_justin(
    years,
    types,
    sectors,
    sizes,
    region
):
    revenue, _ = apply_filters_justin(
        years, types, sectors, sizes, region
    )

    return fig_sector_revenue_justin(revenue)


# Chart 3: client revenue concentration.
@app.callback(
    Output("justin_graph3", "figure"),
    Input("justin_year_range", "value"),
    Input("justin_type", "value"),
    Input("justin_sector", "value"),
    Input("justin_orgsize", "value"),
    Input("justin_region", "value"),
)
def update_graph3_justin(
    years,
    types,
    sectors,
    sizes,
    region
):
    revenue, _ = apply_filters_justin(
        years, types, sectors, sizes, region
    )

    return fig_pareto_justin(revenue)

### Dashboard insight (Elliot)

The growth is real and profitable, but it is being driven by the **clients we already have** rather than the ones we are signing now.

Revenue grew about 8 times between 2021 and 2025, from \\$7.6M to \\$58.8M, and gross margin stayed between 42.7% and 46.0% in every year, so we did not lose profit despite growth(Graph 2). The profit is coming from client age. A client we signed in 2019 to 2021 brought in \\$302k of gross profit in 2025 against \\$136k from one signed in 2024 or 2025, and 86% of all 2025 profit came from clients signed in earlier years. That happened in a year when new signings fell from 44 to 26, so the growth did not depend on winning new clients(Graph 1).

Switching Graph 1 to Revenue per client shows why. Long standing clients pay us \\$679k a year against \\$309k, at almost the same margin of 44.5% against 43.9%. They are giving us more projects, not paying higher prices. Graph 2's *Per client $* view shows what that costs us. Manpower per client rose 78% while revenue per client rose only 69%, and the margin held only because hardware and software cost per client grew more slowly. The flat margin line is the net of two forces pulling against each other, not a sign that nothing is moving.

For an IT solutions integrator, we recommend growing the clients we already have is worth more than chasing new ones, and the real limit on how fast we can grow is people rather than money, because manpower spend grew 8.2 times while client numbers grew only 4.6 times. Using the dashboard filters, this pattern holds for almost every segment we can select. The only one that breaks it is Charity, at 18.6% median margin against 41.8% to 47.3% elsewhere (Graph 3), and at under 1% of revenue that is a pricing decision rather than a threat. Zafran's Graph 1 shows those same clients give us the highest share of promoters at 62.5%, so the ones we earn least from are the ones who like us most.


In [46]:
# ---- PART 5: run the app ----
# jupyter_mode="tab" opens the dashboard in its own browser tab, which gives it
# the whole screen. Press F11 there for a clean full screen screenshot.
if __name__ == "__main__":
    app.run(debug=True, port=8050, jupyter_mode="tab")


Dash app running on http://127.0.0.1:8050/


<IPython.core.display.Javascript object>